<a href="https://colab.research.google.com/github/nicovazzoler/bein-sports-etl-pipeline/blob/main/BeinSports.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Mounted at /content/drive


In [30]:
#------------------------------------------------------------
# CONECTAR AL DRIVE Y A LA API ((AUTENTICACIÓN NATIVA DE COLAB))
#------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')
from google.colab import auth
%reload_ext google.colab.data_table

#-----------------------------------------------------------
# LIBRERÍAS
#------------------------------------------------------------
import pandas as pd
import glob
import numpy as np
from datetime import timedelta
import re
import hashlib
import openpyxl
import gspread
from google.auth import default

#------------------------------------------------------------
# 1. CONFIGURACIÓN Y DICCIONARIO DE COLORES:TIPOS DE TRANSMISIÓN
#------------------------------------------------------------
# Ajustar ruta según el entorno local
archivo_excel = glob.glob('/content/drive/MyDrive/Automatizacion_TV/1_ENTRADA/beIN**') # Buscar el archivo con glob! // Devuelve una lista con cada archivo
MAPA_COLORES = { # Diccionario de colores de las celdas de excel en hexadecimal, calculados con anterioridad. Cada color es un tipo de transmisión
'FF7030A0': "(v)",
 'FFBFBFBF': "(r)",
 'FFFFFF9B': "(d)",
 'FFFFC58A': "A",
 'FFFF8080': "A",
 'FFFF88B1': "A"
}

#------------------------------------------------------------
# 2. CARGA DE DATOS Y COLORES
#------------------------------------------------------------

# Leer datos con Pandas
archivo = pd.read_excel(archivo_excel[0], header=2, skipfooter=1)

# Leer colores con Openpyxl
wb = openpyxl.load_workbook(archivo_excel[0], data_only=True) # Cargar workbook (archivo completo) // [0] para abrir el primero que haya
ws = wb.active # Activar hoja principal

tipos_transmision = [] # Matriz para guardar los tipos de transmisión
fila_inicio = 4 # Como header=2 (fila 3), los datos arrancan en fila 4

# Borrar columnas extras de hora
archivo = archivo.drop(columns = [col for col in archivo if 'ET.' in str(col)]) # Recorre cada columna del df y borra las que contengan "ET." hay muchas ET, pandas las numera .1 .2 y deja ET la primera

# Recorremos cada celda del Excel que corresponde a los datos
for i in range(len(archivo)): # For para recorrer por filas
    fila_tipos = [] # Matriz para guardar los tipos
    fila_excel = fila_inicio + i # Termina de recorrer una fila y pasa a la otra

    for j in range(len(archivo.columns)): # For para recorrer por columnas
        col_excel = j + 1 # openpyxl usa base 1 // Hay que arrancar por la columna 2 porque en la 1 está la hora
        celda = ws.cell(row=fila_excel, column=col_excel) # Llama a la celda, con el nro de fila y de columna

        # Detectar color
        color_hex = str(celda.fill.start_color.index) # Extraer el código de color de la celda, convertirlo a str por las dudas
        tipo = MAPA_COLORES.get(color_hex, None) # Extraer del mapa de colores el color encontrado, si no encuentra que deje vacío
        fila_tipos.append(tipo) # Agregar a la matriz de los tipos

    tipos_transmision.append(fila_tipos) # Agregar a la matriz de tipos_transmision



# Renombrar la columna hora en archivo
archivo.rename(columns={'ET': 'Hora'}, inplace=True) #Renombrar columna a "hora"

# Crear DataFrame de tipos, ponerle el mismo nombre de las columnas del archivo que son las fechas de cada día
df_tipos = pd.DataFrame(tipos_transmision, columns=archivo.columns)

#------------------------------------------------------------
# 3. MELT, UNIÓN Y CREACIÓN COLUMNA DESTACADOS
#------------------------------------------------------------

# Melt Datos
archivo_melt = archivo.melt(id_vars=['Hora'], var_name='Fecha', value_name='Programa')

# Melt Tipos
tipos_melt = df_tipos.melt(id_vars=['Hora'], var_name='Fecha', value_name='tipo_transmision')

# Unir todo en 'archivo_limpio'
archivo_limpio = archivo_melt.copy()
archivo_limpio['tipo_transmision'] = tipos_melt['tipo_transmision']

# Limpiar vacíos en programa
archivo_limpio = archivo_limpio.dropna(subset=['Programa'])




# Crear columna destacado con los valores "A" de tipo_transmision
condicion_destacado = archivo_limpio["tipo_transmision"] == "A"
destacado = archivo_limpio["tipo_transmision"].where(condicion_destacado)
destacado.fillna("", inplace = True)
archivo_limpio["destacado"] = destacado

# Reemplazar los destacados en tipo_transmision
archivo_limpio["tipo_transmision"] = archivo_limpio["tipo_transmision"].str.replace("A", "")

#------------------------------------------------------------
# 4. GENERACIÓN DE HASH
#------------------------------------------------------------
def generar_hash(texto):
    texto = str(texto).strip().lower() # Sacar vacíos y poner todo en minúscula
    return hashlib.md5(texto.encode('utf-8')).hexdigest() # Creación de hash

archivo_limpio['id_hash'] = archivo_limpio['Programa'].apply(generar_hash) # Crear columna hash
# Reordenar para ver Hash y Tipo_transmision al principio
cols = ['id_hash', 'tipo_transmision'] + [c for c in archivo_limpio.columns if c not in ['id_hash', 'tipo_transmision']]
archivo_limpio = archivo_limpio[cols]

#------------------------------------------------------------
# 5. LIMPIEZA DE FECHAS Y HORAS
#------------------------------------------------------------
# Limpiar hora texto incrustada [09:00 PM]
patron_hora = r'\[(\d{2}:\d{2} [AP]M)\]' #Regex

# Reemplazar en HORA
filtro_hora = {"NOON": "12:00 PM", "MIDNIGHT": "12:00 AM"}
archivo_limpio["Hora"] = archivo_limpio["Hora"].replace(filtro_hora, regex = True)


# Filtro booleano con isna() y contains(). isna() busca nulos y devuelve true si encuentra. contains busca coincidencia y si no la encuentra "na" = false
filtro_rescate = archivo_limpio['Hora'].isna() & archivo_limpio['Programa'].str.contains(r'^\[', na=False)

# Usar el filtro en las filas y pararse en la columna hora para guardar contenido // Lo mismo en las filas pero yendo a la columna programa, extrae las horas del patrón []
# Se usa Expand = False para que devuelva una serie y no una tabla
archivo_limpio.loc[filtro_rescate, 'Hora'] = archivo_limpio.loc[filtro_rescate, 'Programa'].str.extract(patron_hora, expand=False)

# Reemplazar la hora que estaba dentro del programa por vacío
archivo_limpio["Programa"] = archivo_limpio["Programa"].str.replace(patron_hora, '', regex=True).str.strip()

# Lógica de fechas (Madrugada)
# Variable temporal que guarda la hora en formato inteligente //  %I = 12hs // %M = minutos // %p = AM/PM - errors = 'coerce' es por si hay algun dato q no sea hora, pone vacío sin error
temp_datetime = pd.to_datetime(archivo_limpio['Hora'], errors='coerce')

# Convertir fecha en formato fecha de pandas
archivo_limpio['Fecha'] = pd.to_datetime(archivo_limpio['Fecha'])
mask_madrugada = temp_datetime.dt.hour < 6 # Se guarda en la mascara el resultado de la expresión booleana. dt.hour extrae solo el número de la hora para la operación booleana
archivo_limpio.loc[mask_madrugada, 'Fecha'] += timedelta(days=1) # Recorre las filas que den True en la máscara de la fila Fecha para sumarle 1 día con timedelta

# Extraer horas y minutos de la hora para sumarselo a la columna de fecha y ahora // Crear columnas de fecha y hora para calcular la duración
horas = pd.to_timedelta(temp_datetime.dt.hour, unit='h')
minutos = pd.to_timedelta(temp_datetime.dt.minute, unit='m')
archivo_limpio ["Timestamp_Completo"] = archivo_limpio["Fecha"] + horas + minutos

# Calcular duración subiendo una fila la columna y restando por la actual (la última quedaría vacía)
archivo_limpio['Duracion'] = (archivo_limpio['Timestamp_Completo'].shift(-1) - archivo_limpio['Timestamp_Completo']).dt.total_seconds() / 60 #Restar hora siguiente con la actual

# Completamos la última restando por el final del día que son las 06:00am
ultimo_timestamp = archivo_limpio['Timestamp_Completo'].iloc[-1]
archivo_limpio.loc[archivo_limpio.index[-1], 'Duracion'] = (ultimo_timestamp.replace(hour=6, minute=0) - ultimo_timestamp).total_seconds() / 60

# Formato int en duración para que no queden decimales
archivo_limpio['Duracion'] = archivo_limpio['Duracion'].astype(int)

# Borramos columna de timestamp
archivo_limpio = archivo_limpio.drop(columns=['Timestamp_Completo'])


#------------------------------------------------------------
# 6. LIMPIEZA DE METADATA
#------------------------------------------------------------

# Función de limpieza para los programas
def limpiar_metadata(programa):
    columna_original = programa # Copia de columna original para guardar como vino al empezar
    texto = str(programa) # Tipo texto

    # Planchar texto
    texto = texto.replace('\n', ' ')
    texto = re.sub(r'\s+', ' ', texto).strip()
    texto = re.sub(r'(20\d{2})/(\d{2})', r'\1-20\2', texto)

    # Temporada
    val_temporada = ""
    match_anio = re.search(r'\b(20\d{2}[-]?20\d{2}|20\d{2})\b', texto) # Busca año en el programa (texto)
    if match_anio:
        val_temporada = match_anio.group(0).strip() # Lo guarda como temporada
        texto = texto.replace(val_temporada, "") # Lo borra del programa

    # Limpieza inicial
    texto = re.sub(r'(?i)\s*-?\s*\b(LIVE|DELAY)\b', '', texto)
    texto = re.sub(r'(?i)\s*Turkish League\s*', '', texto)
    texto = re.sub(r'-+', '-', texto)
    texto = re.sub(r'(?i)^Microprograms\s+', '', texto)
    texto = re.sub(r'(?i)^Boxing\s*-\s*', '', texto)
    texto = re.sub(r'\s*\(([A-Za-z\s]+)\)', r' - \1', texto)
    texto = re.sub(r'\(\d{2}[/.-]\d{2}[/.-]?\d{2,4}?\)', '', texto)
    texto = re.sub(r'(?i)\s*\(Ligue 1 PREVIEW Show\)', '', texto)
    texto = re.sub(r'(?i)Soccer\s+TBD', 'A confirmar', texto)
    texto = re.sub(r'(?i)\s*\[\s*JIP\s*\]', '', texto)
    texto = texto.replace('#', '')
    texto = texto.replace('Match Day', 'Matchday')
    texto = texto.replace("  ", " ")

    # Borrar repetidos
    texto = re.sub(r'(?i)^(.+?)\s+\1$', r'\1', texto)

    # Borrar repetidos con -
    if " - " in texto:
        partes = texto.split(" - ") # Separar por guión en otra matriz
        if len(partes) >= 2 and partes[0].strip().lower() == partes[1].strip().lower(): # Si tiene 2 o mas columnas y las primeras 2 coinciden
            if len(partes) > 2: texto = f"{partes[0].strip()} - {' - '.join(partes[2:])}" # Si tiene mas de 2 guarda la primera, ignora la segunda y conecta el resto con - (aca el join)
            else: texto = partes[0].strip() # Guarda la primera

    val_episodio, val_titulo_capitulo = "", "" # Variales para cada columna

    # Borrar nombre del evento y dejar lo que queda de capítulo
    eventos_edicion = ["Bare Knuckle Boxing BKB",
    "FEI Western European League Show",
    "ONE Friday Fights OFF", "Lux Challenge", "UFC", "BYB", "Bellator", "BRAVE MMA", "MLW FUSION", "Asian Le Mans Series Dubai", "World Archery Highlights" ] # Diccionario de eventos
    evento_detectado = next((e for e in eventos_edicion if texto.lower().startswith(e.lower())), None) # Buscador de eventos


    if evento_detectado:
        patron_evento = re.compile(re.escape(evento_detectado), re.IGNORECASE)
        resto = patron_evento.sub('', texto).strip()
        val_titulo_capitulo = re.sub(r'^[\s-]+', '', resto) # Si quedan guiones o espacios los borra y deja lo que queda de capítulo
        texto = evento_detectado # Dejar el evento limpio en el "texto"
    else: # Programas
        # Buscar "nro capítulo"
        match_ep = re.search(r'(?i)\b(?:Episode|Ep\.?|Capitulo)\s*(\d+)\b', texto) # Todo dígito después de "episodio"
        if match_ep:
            val_episodio = match_ep.group(1).strip() # Guarda lo que venga después de episode en el regex (d+) de nro capítulo
            texto = texto.replace(match_ep.group(0), "") # Borrar todo lo que encuentra en texto

        match_num = re.search(r'-\s*(\d+)$', texto) # Busca guión seguido de un número
        if match_num:
            val_episodio = match_num.group(1).strip() # Guarda solo el dígito (d+)
            texto = re.sub(r'-\s*(\d+)$', '', texto) # Borrar lo encontrado en texto

        # Genero + Instancia
        partes_titulo = [] # Variable temporal para guardar los titulos_capitulos (mas que nada por las instancias)
        match_genero = re.search(r"(?i)(Men['’]?s|Women['’]?s|Mens|Womens)\s+(Final|Semifinal|Quarterfinal|Round)", texto) # Busca capítulos con "género instancia"
        if match_genero:
            encontrado = match_genero.group(0).strip() # Guardar y limpiar todo lo encontrado
            partes_titulo.append(encontrado) # Guardar capítulos en la variable
            texto = texto.replace(encontrado, "") # Reemplazar en el texto

        # Instancia General
        match_inst = re.search(r"(?i)\b(Match\s*Day\s*#?\s*\d*|Quarterfinal\s*\d*|Semifinal\s*\d*|Final|Round of 16|Round\s*\d*|Third Place|Group Stage\s*\d+)\b", texto) # Buscar capítulos con instancia
        if match_inst:
            encontrado_inst = match_inst.group(0).strip() # Guardar y limpiar lo encontrado
            if encontrado_inst and encontrado_inst not in str(partes_titulo): # Filtro de prevención por si encontramos una instancia duplicada
                 partes_titulo.append(encontrado_inst) # Agregar a la lista temp
                 texto = texto.replace(match_inst.group(0), "") # Reemplazar lo encontrado en el título de programa

        # Partido

        match_part = re.search(r'(?i)([a-z0-9\u00C0-\u017F\s\.]+\svs\.?\s[a-z0-9\u00C0-\u017F\s\.]+)', texto) #Buscar partido (tiene que contener "vs.")
        if match_part:
            partes_titulo.insert(0, match_part.group(0).strip()) #Guardar en la variable con 0, para ponerlo primero
            texto = texto.replace(match_part.group(0), "") # Reemplazar en programa

        val_titulo_capitulo = ", ".join(partes_titulo) if partes_titulo else "" # Se guardan todos los capitulos en la variable de columna, uniendo los partidos e instancias con ","
        val_titulo_capitulo = val_titulo_capitulo.replace(" vs ", " vs. ")

    # Últimos reemplazos
    texto = re.sub(r'^[\s-]+|[\s-]+$', '', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    texto = texto.replace(" vs ", " vs. ")

    return pd.Series([texto, val_temporada, val_episodio, val_titulo_capitulo, columna_original]) # Devolver una serie de pandas con todas las columnas creadas



#------------------------------------------------------------
# 7. JUNTAR COLUMNAS PARA TENER EL ARCHIVO LISTO Y BORRAR SALTO DE LINEA EN TITULO_GRILLA
#------------------------------------------------------------

# Limpiar los programas con la función en una variable distinta a archivo
archivo_limpio_2 = archivo_limpio['Programa'].apply(limpiar_metadata)

# Renombrar las columnas
cols = ["titulo_programa", "tit_temp", "nro_cap", "titulo_capitulo", "titulo_grilla"]
archivo_limpio_2.columns = cols

# Conectar los 2 archivos
archivo_limpio_concat = pd.concat([archivo_limpio, archivo_limpio_2], axis=1)

# Borrar columna programa que quedó del archivo de colores
archivo_limpio = archivo_limpio_concat.drop(columns=['Programa'])

# Ordenar columnas
orden_final = [
    'id_hash', 'Fecha', 'Hora', 'Duracion',
    'titulo_programa', 'tit_temp', 'nro_cap', 'titulo_capitulo','destacado', 'tipo_transmision', 'titulo_grilla'
]
archivo_limpio = archivo_limpio[orden_final]

# Reemplazar salto de línea en programa
archivo_limpio["titulo_grilla"] = archivo_limpio["titulo_grilla"].fillna('')
archivo_limpio["titulo_grilla"] = archivo_limpio["titulo_grilla"].astype(str).str.replace('\n', ' // ', regex=False)

print("Archivo procesado con éxito! ")

#------------------------------------------------------------
# 8. CONEXIÓN GOOGLE SHEETS PARA CORRECCIONES MANUALES
#-----------------------------------------------------------

# Conectar a google sheets
print("Conectando con Google Sheets... ")
try:
    # Autenticación y permisos para acceder a la API de Google Drive
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)

    # Variable para abrir la hoja ((revisar nombre del archivo))
    sheet = gc.open('Correcciones_TV').sheet1

    # Guardar todos los registros de la primer hoja del archivo
    worksheet = sheet.get_all_records()

    # Verificar que la hoja no esté vacía
    if worksheet:
        df_corr = pd.DataFrame(worksheet)
        hashes_existentes = df_corr['id_hash'].astype(str).tolist()
    else:
        hashes_existentes = []

    # Hash STR
    archivo_limpio['id_hash'] = archivo_limpio['id_hash'].astype(str)

    # Filtrar para tener los hash únicos
    hash_unico = archivo_limpio.drop_duplicates(subset = ['id_hash'])

    # Buscar los que no estan cargados en el worksheet
    hash_no_cargado = hash_unico[~hash_unico['id_hash'].isin(hashes_existentes)]

    # Cargar datos en el worksheet
    if not hash_no_cargado.empty:
      print(f"Se estan cargando {len(hash_no_cargado)} hashes nuevos en worksheet ")
      filas_a_escribir = []
      for _, row in hash_no_cargado.iterrows(): # Iteramos en cada fila
        # Fila, respetando el orden de las columnas del worksheet
        # id_hash / titulo_programa / tit_temp / nro_cap / titulo_capitulo / titulo_grilla
          fila = [
              row['id_hash'],
                row['titulo_programa'],
                row['tit_temp'],
                row['nro_cap'],
                row['titulo_capitulo'],
                row['titulo_grilla']
            ]
          filas_a_escribir.append(fila)

      # Agregar todas las filas al final del documento
      sheet.append_rows(filas_a_escribir)
      print("Se cargaron datos nuevos en Google Sheets.")

    try:
      sheet.sort((2, 'asc'), (5, 'asc'), range='A2:F1000000')
    except Exception as e:
      print(f"Aviso: No se pudo ordenar el Sheets automáticamente: {e}")

      # Volver a descargar la info actualizada para el merge
      df_corr = pd.DataFrame(sheet.get_all_records())
      df_corr['id_hash'] = df_corr['id_hash'].astype(str)

    # Aplicar correcciones
    if not df_corr.empty:
      # Definir columnas del worksheet y poner tipo STR
      columnas_worksheet = ['titulo_programa', 'tit_temp', 'nro_cap', 'titulo_capitulo', 'titulo_grilla']
      for col in columnas_worksheet:
          if col in df_corr.columns:
              df_corr[col] = df_corr[col].astype(str)
      # Renombramos columnas del sheet
      df_corr = df_corr.rename(columns=lambda x: x + '_corr' if x in columnas_worksheet else x)

      # MERGE de archivo con el sheet --> Suffixies: agrega _manual a las columnas duplicada del sheet
      archivo_limpio = archivo_limpio.merge(df_corr, on='id_hash', how='left', suffixes=('', '_manual'))

      # Columna sheet: Columna archivo
      mapa_correcciones = {
          'titulo_programa': 'titulo_programa',
          'tit_temp': 'tit_temp',
          'nro_cap': 'nro_cap',
          'titulo_capitulo': 'titulo_capitulo',
          'titulo_grilla': 'titulo_grilla'
      }

      # Aplicamos los reemplazos
      for col_sheet, col_archivo in mapa_correcciones.items():
          if col_sheet in archivo_limpio.columns and col_archivo in archivo_limpio.columns:
              # Si la celda de corrección tiene un dato no nulo o vacío, lo reemplaza. Caso contrario lo mantiene
              archivo_limpio[col_archivo] = np.where(
                  archivo_limpio[col_sheet].notna() & (archivo_limpio[col_sheet] != ''),
                  archivo_limpio[col_sheet],
                  archivo_limpio[col_archivo]
              )

      # Limpieza del dataframe
      cols_basura = [c for c in archivo_limpio.columns if c.endswith('_corr') or c == "titulo_grilla"]
      archivo_limpio = archivo_limpio.drop(columns=cols_basura)

      print(f"Correcciones aplicadas desde Google Sheets ")

    else:
        print("Hoja de correcciones vacía, no se aplicaron cambios ")

except Exception as e:
    print(f"Saltando correcciones manuales (Hoja no encontrada o error de conexión): {e}")

#------------------------------------------------------------
# 9. RETOQUES FINALES
#------------------------------------------------------------
# Agregar columna N
if 'N' not in archivo_limpio.columns:
  archivo_limpio.insert(0, 'N', range(1, len(archivo_limpio) + 1))

# Crear columna día
dias_espanol = {
    0: 'Lunes', 1: 'Martes', 2: 'Miércoles',
    3: 'Jueves', 4: 'Viernes', 5: 'Sábado', 6: 'Domingo'
}
# dt.dayofweek devuelve un número de 0 a 6. El .map() lo traduce al diccionario.
archivo_limpio['dia'] = archivo_limpio['Fecha'].dt.dayofweek.map(dias_espanol)

# Formato fecha DD-MM-YYYY
archivo_limpio['Fecha'] = archivo_limpio['Fecha'].dt.strftime('%d-%m-%Y')

# Filtramos y ordenamos el DataFrame solo con las columnas definitivas requeridas para la salida
df_final = archivo_limpio[['N', 'dia', 'Fecha', 'Hora', 'Duracion', 'titulo_programa',
                            'tit_temp', 'nro_cap', 'titulo_capitulo', 'destacado', 'tipo_transmision']].copy()
# Columnas con numeros
columnas_numericas = ['N', 'nro_cap']
for col in columnas_numericas:
    if col in df_final.columns:
        # Convertir en formato Int64 (enteros)
        # Caso que no sean enteros, los ignora
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce').astype('Int64')


# Tit_temp
def limpiar_mixto(valor):
  if str(valor).isnumeric():
    return int(valor)
  return valor
if 'tit_temp' in df_final.columns:
  df_final['tit_temp'] = df_final['tit_temp'].apply(limpiar_mixto)


# Nombre del archivo
nombre_salida = "grilla_limpia_final.xlsx"

# Exportar Excel sin índice de Pandas
carpeta = f"/content/drive/MyDrive/Automatizacion_TV/2_SALIDA/{nombre_salida}"
df_final.to_excel(carpeta, index=False)

print(f"Archivo '{nombre_salida}' generado con éxito.")

df_final.head(10)









Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipython-input-723/2138670766.py:141: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_datetime = pd.to_datetime(archivo_limpio['Hora'], errors='coerce')


Archivo procesado con éxito! 
Conectando con Google Sheets... 
Se estan cargando 110 hashes nuevos en worksheet 
Se cargaron datos nuevos en Google Sheets.
Correcciones aplicadas desde Google Sheets 
Archivo 'grilla_limpia_final.xlsx' generado con éxito.


,N,dia,Fecha,Hora,Duracion,titulo_programa,tit_temp,nro_cap,titulo_capitulo,destacado,tipo_transmision
0,1,Sábado,28-02-2026,06:00 AM,30,Paid Programming,,<NA>,,,None
1,2,Sábado,28-02-2026,06:30 AM,30,Paid Programming,,<NA>,,,None
2,3,Sábado,28-02-2026,07:00 AM,30,Paid Programming,,<NA>,,,None
3,4,Sábado,28-02-2026,07:30 AM,30,Paid Programming,,<NA>,,,None
4,5,Sábado,28-02-2026,08:00 AM,30,The Ligue 1 Show,2025-2026,<NA>,,,(r)
5,6,Sábado,28-02-2026,08:30 AM,120,Ligue 1,,<NA>,"Strasbourg vs. Lens, Matchday 24",,(r)
6,7,Sábado,28-02-2026,10:30 AM,20,Ligue 1 Show,,<NA>,,,(r)
7,8,Sábado,28-02-2026,10:50 AM,100,Ligue 1,,<NA>,"Stade Rennes vs. Toulouse, Matchday 24",,(v)
8,9,Sábado,28-02-2026,12:30 PM,150,Ligue 1,,<NA>,"AS Monaco vs. Angers, Matchday 24",,(v)
9,10,Sábado,28-02-2026,03:00 PM,120,Ligue 1,,<NA>,"Le Havre vs. Paris Saint Germain, Matchday 24",,(v)
